In [1]:
import torch 
import torch.nn as nn
from torchvision.transforms import v2
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torchsummary import summary
from torchmetrics import Accuracy

from tqdm import tqdm
import os
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

In [2]:
def loadDataset(res, batch_size, datasetType):
    base_dir = os.getcwd()
    folder_dir = os.path.join(base_dir, "processed_data", datasetType)
    
    transformation = v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True), # Convert image to float ranging from 0 - 1
        v2.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5)),
        v2.RandomResizedCrop((224,224), (0.08, 1), (0.75, 1.25)),
        v2.RandomHorizontalFlip(),
        v2.Resize((res, res)),
    ])

    dataset = ImageFolder(folder_dir, transform=transformation)

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    return loader

In [3]:
## Building model

weights = EfficientNet_B0_Weights.DEFAULT

efficientnet =  efficientnet_b0(weights=weights)

for parameter in efficientnet.features.parameters():
    parameter.requires_grad = False

class EfficientNetFER(nn.Module):
    def __init__(self):
        super(EfficientNetFER, self).__init__()

        self.layers = nn.Sequential(
            efficientnet,
            nn.Linear(1000, 7),
            nn.Softmax(dim=0)
        )
    
    def forward(self, x):
        return self.layers(x)

efficientnet_FER = EfficientNetFER()

summary(efficientnet_FER, (3,224,224), device='cpu')

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 112, 112]             864
       BatchNorm2d-2         [-1, 32, 112, 112]              64
              SiLU-3         [-1, 32, 112, 112]               0
            Conv2d-4         [-1, 32, 112, 112]             288
       BatchNorm2d-5         [-1, 32, 112, 112]              64
              SiLU-6         [-1, 32, 112, 112]               0
 AdaptiveAvgPool2d-7             [-1, 32, 1, 1]               0
            Conv2d-8              [-1, 8, 1, 1]             264
              SiLU-9              [-1, 8, 1, 1]               0
           Conv2d-10             [-1, 32, 1, 1]             288
          Sigmoid-11             [-1, 32, 1, 1]               0
SqueezeExcitation-12         [-1, 32, 112, 112]               0
           Conv2d-13         [-1, 16, 112, 112]             512
      BatchNorm2d-14         [-1, 16, 1

In [4]:
## Testing for whether CUDA is available for training

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device('cpu')
    print("CUDA not available. Using CPU.")

Using GPU: NVIDIA GeForce RTX 4060 Ti


In [5]:
## Load dataset

train_dataloader = loadDataset(224, 32, "train")
test_dataloader = loadDataset(224, 32, "test")
val_dataloader = loadDataset(224, 32, "validation")

## Setup loss function, acc function, and optmizer

lossFn = nn.CrossEntropyLoss()
accFn = Accuracy('multiclass', num_classes=7).to(device)
optimizer = torch.optim.SGD(efficientnet_FER.parameters(), lr=0.001, momentum=0.89)


In [6]:
torch.manual_seed(42) # 42 is the answer to everything!
    
epochs = 12

efficientnet_FER = efficientnet_FER.to(device)


In [7]:
# Training

for epoch in range(0, epochs):

    print(f"\nEpoch {epoch+1} / {epochs}: ---------------------------")

    train_loss, train_acc = 0, 0

    for batch, (X, y) in enumerate(tqdm(train_dataloader, desc="Training Model on Num Batches")):
        efficientnet_FER.train()

        X, y = X.to(device), y.to(device)

        # 1. Forward pass
        y_pred = efficientnet_FER(X) # Still in logits from (probabilities)

        # 2. Calculate loss (per batch)
        loss = lossFn(y_pred, y)
        train_loss += loss
        train_acc += accFn(y_pred.argmax(dim=1), y)

        # 3. Optimizer zero grad
        optimizer.zero_grad()

        # 4. Loss backward
        loss.backward()

        # 5. Optimizer step
        optimizer.step()
        
    # Divide total train loss and accuracy by length of train dataloader (average loss per batch per epoch)
    train_loss /= len(train_dataloader)
    train_acc /= len(train_dataloader)
    train_acc *= 100

    print(f"\nTrain loss: {train_loss:.4f}, Train acc: {train_acc:.2f}%")

print("\n-------Model has finished training-------\n")


Epoch 1 / 12: ---------------------------


Training Model on Num Batches: 100%|██████████| 898/898 [01:36<00:00,  9.30it/s]



Train loss: 1.9451, Train acc: 15.52%

Epoch 2 / 12: ---------------------------


Training Model on Num Batches: 100%|██████████| 898/898 [01:31<00:00,  9.86it/s]



Train loss: 1.9415, Train acc: 19.32%

Epoch 3 / 12: ---------------------------


...


Train loss: 1.9351, Train acc: 22.02%

Epoch 5 / 12: ---------------------------


Training Model on Num Batches: 100%|██████████| 898/898 [01:36<00:00,  9.33it/s]



Train loss: 1.9325, Train acc: 22.22%

Epoch 6 / 12: ---------------------------


Training Model on Num Batches:  13%|█▎        | 121/898 [00:12<01:17, 10.05it/s]


KeyboardInterrupt: 

In [46]:
# Testing

test_loss, test_acc = 0, 0
efficientnet_FER.eval()

with torch.inference_mode():
    for batch, (X, y) in enumerate(tqdm((test_dataloader), desc="Testing Model")):
        
        X, y = X.to(device), y.to(device)

        # 1. Forward pass
        test_pred = efficientnet_FER(X)

        # 2. Calculate loss (accumulatively)
        test_loss += lossFn(test_pred, y)

        # 3. Calculate accuracy (preds need to be same as y)
        test_acc += accFn(test_pred.argmax(dim=1), y)
    
    # Divide total test loss by length of test dataloader (per batch)
    test_loss /= len(test_dataloader)

    # Divide total accuracy by length of test dataloader (per batch)
    test_acc /= len(test_dataloader)

    # Times 100 to make it into a percentage
    test_acc *= 100

print(f"Test loss: {test_loss:.4f}, Test acc: {test_acc:.2f}%\n")

Testing Model: 100%|██████████| 113/113 [00:08<00:00, 13.61it/s]

Test loss: 1.9099, Test acc: 12.98%

